# OpenEnv + TRL Minimal Colab\n\nThis notebook provides a minimal reproducible training loop for the hackathon requirement using Hugging Face TRL.\n\nPipeline:\n1. Generate trajectories from `SatyaEnv`\n2. Convert to prompt-completion pairs\n3. Run lightweight TRL SFT trainer\n4. Save model + training logs

In [ ]:
!pip -q install -U transformers datasets accelerate trl peft bitsandbytes sentencepiece

In [ ]:
import os\nimport json\nfrom datasets import Dataset\n\n# Clone your repo in Colab before this step or mount Drive and set repo path.\nREPO_PATH = '/content/multi-agent'\nos.chdir(REPO_PATH)\n\nos.environ['TRAINING_AGENT_MODE'] = 'rl'\nos.environ['SELF_IMPROVEMENT_ENABLED'] = 'true'\nos.environ['NUM_EPISODES'] = '20'\nos.environ['CURRICULUM_PHASE_EPISODES'] = '5'\n\n!python train.py

In [ ]:
with open('results/training_results.json', 'r', encoding='utf-8') as f:\n    episodes = json.load(f)\n\nsamples = []\nfor ep in episodes:\n    for step in ep.get('steps', []):\n        acts = step.get('actions', {})\n        for agent_id, a in acts.items():\n            prompt = f"Agent={agent_id}; Hour={step.get('hour')}; Choose resource action."\n            completion = json.dumps(a)\n            samples.append({'prompt': prompt, 'completion': completion})\n\ndataset = Dataset.from_list(samples)\ndataset = dataset.train_test_split(test_size=0.1, seed=42)\ndataset

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM\nfrom trl import SFTConfig, SFTTrainer\n\nmodel_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'\ntokenizer = AutoTokenizer.from_pretrained(model_id)\nif tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token\n\ndef to_text(example):\n    return {'text': f"<|user|>{example['prompt']}<|assistant|>{example['completion']}"}\n\ntrain_ds = dataset['train'].map(to_text)\neval_ds = dataset['test'].map(to_text)\n\nmodel = AutoModelForCausalLM.from_pretrained(model_id)\n\ncfg = SFTConfig(\n    output_dir='trl_sft_satya',\n    per_device_train_batch_size=2,\n    per_device_eval_batch_size=2,\n    num_train_epochs=1,\n    logging_steps=10,\n    evaluation_strategy='steps',\n    eval_steps=20,\n    save_steps=50,\n    learning_rate=2e-5,\n)\n\ntrainer = SFTTrainer(\n    model=model,\n    args=cfg,\n    train_dataset=train_ds,\n    eval_dataset=eval_ds,\n    processing_class=tokenizer,\n    formatting_func=lambda x: x['text'],\n)\n\ntrainer.train()\ntrainer.save_model('trl_sft_satya/final_model')\ntokenizer.save_pretrained('trl_sft_satya/final_model')

In [ ]:
print('Artifacts:')\n!ls results\n!ls trl_sft_satya/final_model